# LMTA ALMP XML Export: Clean Start

**Notebook engine:** PySpark.

This notebook uses the PySpark runtime only for Microsoft Fabric integration and `notebookutils`. It does not use Spark DataFrames. Warehouse access, XML construction, XSD validation, and Lakehouse file output are ordinary Python.

Use only the new module name `lmta_almp_export.py`. Do not upload or import `almp_xml_export.py` in this notebook.


In [ ]:
%pip install lxml pyodbc pandas


If Fabric restarts the session after package installation, continue with the next cell.


In [ ]:
from pathlib import Path
import sys
import importlib
import json

MODULE_DIR = Path("/lakehouse/default/Files/LMTA/code/python")
MODULE_PATH = MODULE_DIR / "lmta_almp_export.py"

if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Upload the new module to: {MODULE_PATH}")

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# Unique module name prevents collision with all older almp_xml_export versions.
sys.modules.pop("lmta_almp_export", None)
importlib.invalidate_caches()
import lmta_almp_export

print("Loaded from:", lmta_almp_export.__file__)
print("Module version:", lmta_almp_export.MODULE_VERSION)
print("ExportConfig available:", hasattr(lmta_almp_export, "ExportConfig"))
print("test_connection available:", hasattr(lmta_almp_export, "test_connection"))
print("run_export available:", hasattr(lmta_almp_export, "run_export"))

ExportConfig = lmta_almp_export.ExportConfig
test_connection = lmta_almp_export.test_connection
run_export = lmta_almp_export.run_export


In [ ]:
# EDIT THESE VALUES ONLY
SUBMISSION_ID = 101
SQL_SERVER = "REPLACE-WITH-ACTUAL-ENDPOINT.datawarehouse.fabric.microsoft.com"
SQL_DATABASE = "LMTA"
PROGRAM_TYPE = "LMDA"       # LMDA or WDA
SEQUENCE_NUMBER = 1
MAIN_XSD_NAME = "ALMP.xsd"

# Production requirement: populate exact XSD xs:sequence order.
ELEMENT_ORDER = {
    # "submission": [...],
    # "sys_action": [...],
    # "client": [...],
    # "action_plan": [...],
    # "followup": [...],
    # "intervention": [...],
    # "support": [...],
}

config = ExportConfig(
    submission_id=SUBMISSION_ID,
    sql_server=SQL_SERVER,
    sql_database=SQL_DATABASE,
    program_type=PROGRAM_TYPE,
    sequence_number=SEQUENCE_NUMBER,
    main_xsd_name=MAIN_XSD_NAME,
    table_element_order=ELEMENT_ORDER,
)

print("Configuration created for submission:", config.submission_id)


In [ ]:
# Run this before XML generation.
connection_result = test_connection(config)
print(json.dumps(connection_result, indent=2, default=str))


In [ ]:
# Run only after the connection test succeeds and SUBMISSION_ID exists.
result = run_export(config)
print(json.dumps(result, indent=2, default=str))


## Required ICT_DL1 locations

- Module: `Files/LMTA/code/python/lmta_almp_export.py`
- Main XSD: `Files/LMTA/reference/xsd/current/ALMP.xsd`
- Imported XSDs: same `current` folder
- Valid XML: `Files/LMTA/output/xml/submitted/`
- Invalid XML: `Files/LMTA/output/xml/rejected/`
- Validation reports: `Files/LMTA/output/validation/`
